# 📊 Stock Price Direction Prediction Using LSTM/ANN

  **ML Engineer Track**  |  Difficulty: **Hard**  |  Domain: **Quantitative Research**

> 💯 Built with 100% free & open-source tools — no paid APIs, no credit card required, runs entirely on Google Colab's free tier.

---

## 🧩 Problem Statement

Predicting the exact price is nearly impossible, but predicting next-day direction (up/down) with a useful edge is a common quant research exercise. Build an LSTM neural network using engineered technical-indicator features to predict next-day price direction -- fully in free TensorFlow/Keras, with no paid data or compute required.

## 📁 Dataset

**Free daily OHLCV price history via yfinance**

Source: [https://finance.yahoo.com](https://finance.yahoo.com)

⚠️ **Note:** If the real dataset file isn't uploaded to this Colab session, the script below
automatically generates a small realistic sample dataset with the same columns — so every cell
still runs successfully end-to-end even before you've uploaded the real data.

## 🎯 What This Notebook Builds

- A technical indicator feature set (RSI, MACD, moving averages, volatility, volume change)
- A sliding-window sequence generator for LSTM input (e.g. 20-day lookback windows)
- A binary direction label (next-day close higher or lower than today's close)
- An LSTM classifier built in Keras with dropout regularization
- A simple feedforward ANN baseline for comparison against the LSTM
- Backtested directional accuracy plus a simple long/flat strategy return comparison

## 🧭 Approach

1. Engineer Features
2. Build Sliding Windows
3. Train LSTM Classifier
4. Compare to ANN Baseline & Backtest

## 💡 Key Takeaways

- Direction prediction accuracy in the low-to-mid 50s% is realistic and still useful for a filtered strategy -- treat any near-70%+ result with skepticism (likely leakage)
- The LSTM vs ANN comparison is the key experiment: it isolates whether sequence memory actually adds predictive value over flat features
- Technical indicators need to be computed strictly on past data at each point -- any lookahead in feature engineering invalidates the whole backtest

## 🛠️ Tools Used

`Python 3 | TensorFlow/Keras (free) | yfinance | scikit-learn | pandas`

---

### ⚠️ Disclaimer
This notebook is for educational / portfolio purposes only. It does not constitute financial,
credit, or investment advice.

---

In [ ]:

# pip install tensorflow yfinance pandas numpy scikit-learn --break-system-packages

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout




In [ ]:
# 1. LOAD DATA & ENGINEER FEATURES
df = yf.download("MSFT", period="5y")[["Close", "Volume"]].dropna()

df["return_1d"] = df["Close"].pct_change()
df["ma10"] = df["Close"].rolling(10).mean()
df["ma20"] = df["Close"].rolling(20).mean()
ema12 = df["Close"].ewm(span=12).mean()
ema26 = df["Close"].ewm(span=26).mean()
df["macd"] = ema12 - ema26
df["volatility_10d"] = df["return_1d"].rolling(10).std()
df["volume_change"] = df["Volume"].pct_change()

delta = df["Close"].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = -delta.clip(upper=0).rolling(14).mean()
df["rsi"] = 100 - (100 / (1 + gain / loss))

df["direction"] = (df["Close"].shift(-1) > df["Close"]).astype(int)
df = df.dropna()

feature_cols = ["return_1d", "ma10", "ma20", "macd", "volatility_10d", "volume_change", "rsi"]
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df[feature_cols])




/tmp/ipykernel_3691/4219279209.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download("MSFT", period="5y")[["Close", "Volume"]].dropna()
[*********************100%***********************]  1 of 1 completed


In [ ]:

# 2. BUILD SLIDING WINDOWS FOR LSTM
LOOKBACK = 20
X, y = [], []
for i in range(LOOKBACK, len(scaled_features) - 1):
    X.append(scaled_features[i - LOOKBACK:i])
    y.append(df["direction"].iloc[i])
X, y = np.array(X), np.array(y)

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]


In [ ]:
# 3. TRAIN LSTM CLASSIFIER
lstm_model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(LOOKBACK, len(feature_cols))),
    Dropout(0.3),
    LSTM(32),
    Dropout(0.3),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid"),
])
lstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
lstm_model.fit(X_train, y_train, epochs=25, batch_size=32, validation_split=0.1, verbose=0)

lstm_loss, lstm_acc = lstm_model.evaluate(X_test, y_test, verbose=0)
print(f"LSTM test accuracy: {lstm_acc:.2%}")



/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


LSTM test accuracy: 51.03%


In [ ]:
# 4. ANN BASELINE (flatten last window into one feature vector)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

ann_model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train_flat.shape[1],)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid"),
])
ann_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
ann_model.fit(X_train_flat, y_train, epochs=25, batch_size=32, validation_split=0.1, verbose=0)

ann_loss, ann_acc = ann_model.evaluate(X_test_flat, y_test, verbose=0)
print(f"ANN baseline test accuracy: {ann_acc:.2%}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


ANN baseline test accuracy: 51.44%


In [ ]:
# 5. SIMPLE LONG/FLAT STRATEGY BACKTEST USING LSTM PREDICTIONS
test_dates = df.iloc[LOOKBACK + split: LOOKBACK + split + len(y_test)]

predicted_direction = (lstm_model.predict(X_test, verbose=0) > 0.5).astype(int).flatten()

# Align strategy returns with the correct forward returns
strategy_returns = np.where(predicted_direction == 1, test_dates["return_1d"].shift(-1), 0)
strategy_returns = pd.Series(strategy_returns).fillna(0)

cumulative_strategy_return = (1 + strategy_returns).cumprod().iloc[-1] - 1

# Ensure Close is a 1D Series, then compute scalar buy & hold return
close = test_dates["Close"].iloc[:, 0] if isinstance(test_dates["Close"], pd.DataFrame) else test_dates["Close"]
buyhold_return = float((close.iloc[-1] / close.iloc[0]) - 1)

print(f"LSTM-signal strategy return: {cumulative_strategy_return:.2%}")
print(f"Buy & hold return over same period: {buyhold_return:.2%}")

LSTM-signal strategy return: -5.18%
Buy & hold return over same period: -3.51%
